# 15 — 50/50-blanding av physics-modellene

**Gruppe 119 — Henrik Remmen (541747)**  
**Kaggle-lag:** `[119] SINTEF Databreach`

Kjør kodecellen under for å lage `submissions/physics_blend_50_50.csv` fra de to ferdige prediksjonsfilene. Den bruker ikke fasitdata og trener ingen modeller.

Dette er en rask test av blandingen. Hvis den velges som endelig Kaggle-innsending, må vi lage en egen kort notebook som trener begge modellene fra rådata, fordi ferdige CSV-filer ikke er tilgjengelige når professoren kjører innleveringen.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'sample_submission.csv').exists()
     and (path / 'submissions').is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Åpne notebooken fra sintef_project eller notebooks-mappen.')

xgb_path = ROOT / 'submissions/xgb_physics_v2.csv'
lgbm_path = ROOT / 'submissions/lgbm_physics_v2.csv'
for path in (xgb_path, lgbm_path):
    if not path.is_file():
        raise FileNotFoundError(f'Mangler modellens prediksjonsfil: {path}')

sample = pd.read_csv(ROOT / 'sample_submission.csv', usecols=['Run No'])
xgb = pd.read_csv(xgb_path)
lgbm = pd.read_csv(lgbm_path)
expected_columns = pd.read_csv(ROOT / 'sample_submission.csv', nrows=0).columns.tolist()
if xgb.columns.tolist() != expected_columns or lgbm.columns.tolist() != expected_columns:
    raise ValueError('Kolonnene eller rekkefølgen matcher ikke sample_submission.csv.')
if not xgb['Run No'].equals(sample['Run No']) or not lgbm['Run No'].equals(sample['Run No']):
    raise ValueError('Run No eller radrekkefølgen matcher ikke sample_submission.csv.')

prediction_columns = expected_columns[1:]
xgb_values = xgb[prediction_columns].to_numpy(dtype=np.float64)
lgbm_values = lgbm[prediction_columns].to_numpy(dtype=np.float64)
if not np.isfinite(xgb_values).all() or not np.isfinite(lgbm_values).all():
    raise ValueError('En av modellfilene inneholder NaN eller uendelig verdi.')
if min(xgb_values.min(), lgbm_values.min()) < 0 or max(xgb_values.max(), lgbm_values.max()) > 1:
    raise ValueError('En av modellfilene har prediksjoner utenfor [0, 1].')

blend = xgb.copy()
blend[prediction_columns] = 0.5 * xgb_values + 0.5 * lgbm_values
output_path = ROOT / 'submissions/physics_blend_50_50.csv'
blend.to_csv(output_path, index=False)
print('Klar til opplasting på Kaggle:', output_path)
print('Antall rader:', len(blend), '| antall prediksjonskolonner:', len(prediction_columns))
display(blend.iloc[:3, :8])

Klar til opplasting på Kaggle: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/submissions/physics_blend_50_50.csv
Antall rader: 725 | antall prediksjonskolonner: 2352


,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6
0,2923,0.108930,0.068554,0.039475,0.029846,0.020315,0.017901,0.008322
1,2924,0.868713,0.866741,0.885165,0.884662,0.884848,0.909455,0.925478
2,2925,0.999437,0.999460,0.999455,0.998925,0.999478,0.999556,0.999612
